In [28]:
import subprocess
import sys

In [29]:
from pathlib import Path
import pandas as pd

# Load the raw tables
DATA_DIR = Path("/home/gandalf/Documents/Data/Deep Mut")
RNASEQ_PATH = DATA_DIR / "rnaseq_data.tsv"
META_PATH = DATA_DIR / "meta_data.tsv"

# Load metadata normally
meta_df = pd.read_csv(META_PATH, sep="\t")

# Read first row for gene names
header_df = pd.read_csv(RNASEQ_PATH, sep="\t", header=None, nrows=1, dtype=str)
gene_names = header_df.iloc[0, :].tolist()

# Read data rows (skip header) with error handling
rnaseq_df = pd.read_csv(RNASEQ_PATH, sep="\t", header=None, skiprows=1, dtype=str, engine='python', on_bad_lines='warn')

# Pad/truncate to match header length
if rnaseq_df.shape[1] > len(gene_names):
    rnaseq_df = rnaseq_df.iloc[:, :len(gene_names)]
elif rnaseq_df.shape[1] < len(gene_names):
    for i in range(len(gene_names) - rnaseq_df.shape[1]):
        rnaseq_df[len(rnaseq_df.columns)] = None

rnaseq_df.columns = gene_names

# Rename first column to "sample_id"
rnaseq_df = rnaseq_df.rename(columns={gene_names[0]: 'sample_id'})

print(f"RNA-seq shape: {rnaseq_df.shape}")

RNA-seq shape: (9626, 19311)


In [30]:
sorted(meta_df["histological_grade"].dropna().unique())

['G1',
 'G2',
 'G3',
 'G4',
 'GB',
 'GX',
 'High Grade',
 'Low Grade',
 '[Discrepancy]',
 '[Unknown]']

In [31]:
import re

TCGA_ID_PATTERN = re.compile(r"TCGA-[A-Z0-9]+-[A-Z0-9]+-\d+")


def _clean_meta(df: pd.DataFrame) -> pd.DataFrame:
    meta = df.copy()
    meta.columns = (
        meta.columns.astype(str)
        .str.strip()
        .str.strip('"')
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )
    
    # Use 'sample' column as the patient ID directly (no normalization)
    if 'sample' in meta.columns:
        meta['patient_id'] = meta['sample'].astype(str).str.strip().str.replace('"', '')
    else:
        raise ValueError("'sample' column not found in metadata")
    
    meta = meta.dropna(subset=['patient_id']).drop_duplicates('patient_id')

    if "histological_grade" in meta.columns:
        meta["histological_grade"] = (
            meta["histological_grade"].astype(str).str.strip().str.upper().replace({"": pd.NA})
        )

    return meta


def _clean_rnaseq(df: pd.DataFrame) -> pd.DataFrame:
    """
    RNA-seq data structure:
    - Header row: gene names (19311 fields)
    - Data rows: TCGA sample ID in col 1, then 19310 expression values
    """
    # Extract gene names from header
    gene_names = df.columns.astype(str).str.strip().str.replace('"', '')
    
    # Extract sample IDs from first column (no normalization)
    sample_ids = df.iloc[:, 0].astype(str).str.strip().str.replace('"', '')
    
    # Extract expression data: columns 2 onwards
    expr_data = df.iloc[:, 1:].copy()
    expr_data.columns = gene_names[:len(expr_data.columns)]
    expr_data = expr_data.apply(pd.to_numeric, errors="coerce")
    
    # Create dataframe with sample_id and gene expressions
    rna_clean = pd.concat([
        sample_ids.reset_index(drop=True),
        expr_data.reset_index(drop=True)
    ], axis=1)
    
    rna_clean.columns = ['patient_id'] + list(expr_data.columns)
    
    # Remove any empty sample IDs
    rna_clean = rna_clean[rna_clean['patient_id'].str.len() > 0].dropna(subset=['patient_id'])
    
    # Pivot: genes as rows, patients as columns
    rna_clean = rna_clean.set_index('patient_id')
    
    # Handle duplicate patients by averaging
    rna_final = rna_clean.groupby(level=0).mean()
    rna_final = rna_final.T  # genes x patients
    rna_final['gene_id'] = rna_final.index
    rna_final = rna_final[['gene_id'] + [c for c in rna_final.columns if c != 'gene_id']]
    rna_final = rna_final.reset_index(drop=True)
    
    return rna_final


clean_meta_df = _clean_meta(meta_df)
clean_rnaseq_df = _clean_rnaseq(rnaseq_df)

rnaseq_patient_ids = set(clean_rnaseq_df.columns) - {"gene_id"}
matched_patient_ids = sorted(set(clean_meta_df["patient_id"]) & rnaseq_patient_ids)

clean_meta_df = clean_meta_df[clean_meta_df["patient_id"].isin(matched_patient_ids)].reset_index(drop=True)
clean_rnaseq_df = clean_rnaseq_df[["gene_id"] + matched_patient_ids]

print(f"✓ Matched {len(matched_patient_ids)} patients across both tables")
print(f"  - Metadata: {len(clean_meta_df)} rows")
print(f"  - RNA-seq: {len(clean_rnaseq_df)} genes × {len(matched_patient_ids)} samples")
clean_meta_df.head()

✓ Matched 9626 patients across both tables
  - Metadata: 9626 rows
  - RNA-seq: 19310 genes × 9626 samples


,sample_type_id,sample_type,project_id,rnaseqid,mutid,sample,x_patient,cancer.type.abbreviation,age_at_initial_pathologic_diagnosis,gender,...,os.time,dss,dss.time,dfi,dfi.time,pfi,pfi.time,redaction,x_primary_disease,patient_id
0,1,Primary Tumor,TCGA-GBM,TCGA-02-0047-01A,TCGA-02-0047-01,TCGA-02-0047-01,TCGA-02-0047,GBM,78.0,MALE,...,448.0,1.0,448.0,NaN,NaN,1.0,57.0,NaN,glioblastoma multiforme,TCGA-02-0047-01
1,1,Primary Tumor,TCGA-GBM,TCGA-02-0055-01A,TCGA-02-0055-01,TCGA-02-0055-01,TCGA-02-0055,GBM,62.0,FEMALE,...,76.0,1.0,76.0,NaN,NaN,1.0,6.0,NaN,glioblastoma multiforme,TCGA-02-0055-01
2,1,Primary Tumor,TCGA-GBM,TCGA-02-2483-01A,TCGA-02-2483-01,TCGA-02-2483-01,TCGA-02-2483,GBM,43.0,MALE,...,466.0,0.0,466.0,NaN,NaN,0.0,466.0,NaN,glioblastoma multiforme,TCGA-02-2483-01
3,1,Primary Tumor,TCGA-GBM,TCGA-02-2485-01A,TCGA-02-2485-01,TCGA-02-2485-01,TCGA-02-2485,GBM,53.0,MALE,...,470.0,0.0,470.0,NaN,NaN,1.0,186.0,NaN,glioblastoma multiforme,TCGA-02-2485-01
4,1,Primary Tumor,TCGA-GBM,TCGA-02-2486-01A,TCGA-02-2486-01,TCGA-02-2486-01,TCGA-02-2486,GBM,64.0,MALE,...,618.0,1.0,618.0,NaN,NaN,1.0,618.0,NaN,glioblastoma multiforme,TCGA-02-2486-01


In [32]:
print("Test _select_patient_id function:")
meta_test = meta_df.copy()
cols_lower = (
    meta_test.columns.astype(str)
    .str.strip()
    .str.strip('"')
    .str.lower()
    .str.replace(" ", "_", regex=False)
)
meta_test.columns = cols_lower

id_candidates = ['sample', 'x_patient', 'rnaseqid', 'sample_type_id', 'mutid']

def _normalize_patient_id(val):
    if pd.isna(val):
        return pd.NA
    s = str(val).strip().replace('"', '')
    return s if s else pd.NA

def _select_patient_id_test(row):
    for col in id_candidates:
        val = row.get(col)
        normalized = _normalize_patient_id(val)
        print(f"    {col}: {val} -> {normalized}")
        if not pd.isna(normalized):
            return normalized
    return pd.NA

# Test on first row
print("First row:")
result = _select_patient_id_test(meta_test.iloc[0])
print(f" => {result}")

print("\nSecond row:")
result = _select_patient_id_test(meta_test.iloc[1])
print(f" => {result}")

Test _select_patient_id function:
First row:
    sample: TCGA-02-0047-01 -> TCGA-02-0047-01
 => TCGA-02-0047-01

Second row:
    sample: TCGA-02-0055-01 -> TCGA-02-0055-01
 => TCGA-02-0055-01


In [33]:
print("Meta patient_id (first 5):")
print(clean_meta_df['patient_id'].head().tolist())

print("\nRNA-seq patient_id (first 5):")
print(list(rnaseq_patient_ids)[:5])

print(f"\nMeta unique count: {len(set(clean_meta_df['patient_id']))}")
print(f"RNA-seq unique count: {len(rnaseq_patient_ids)}")

print("\nIntersection:")
intersection = set(clean_meta_df["patient_id"]) & rnaseq_patient_ids
print(f"Matched: {len(intersection)}")
if intersection:
    print(f"Examples: {list(intersection)[:5]}")
else:
    # Show a few from each
    print(f"Meta sample: {clean_meta_df['patient_id'].iloc[0] if len(clean_meta_df) > 0 else 'N/A'}")
    print(f"RNA sample: {list(rnaseq_patient_ids)[0] if rnaseq_patient_ids else 'N/A'}")

Meta patient_id (first 5):
['TCGA-02-0047-01', 'TCGA-02-0055-01', 'TCGA-02-2483-01', 'TCGA-02-2485-01', 'TCGA-02-2486-01']

RNA-seq patient_id (first 5):
['TCGA-A3-3357-01', 'TCGA-MQ-A6BR-01', 'TCGA-YU-A912-01', 'TCGA-3U-A98I-01', 'TCGA-29-1705-01']

Meta unique count: 9626
RNA-seq unique count: 9626

Intersection:
Matched: 9626
Examples: ['TCGA-A3-3357-01', 'TCGA-MQ-A6BR-01', 'TCGA-YU-A912-01', 'TCGA-3U-A98I-01', 'TCGA-29-1705-01']


In [44]:
print("=== DATASET BEFORE LABELING ===")
print(f"Metadata shape: {clean_meta_df.shape}")
print(f"RNA-seq shape: {clean_rnaseq_df.shape}")

# Map cancer grades to binary risk labels
# Reverted to prior definition used for the ~0.88 accuracy runs:
# low risk: G1 / LOW GRADE
# high risk: G3 / G4 / HIGH GRADE
# (G2 and unknowns are excluded)
low_risk_grades = {'G1', 'LOW GRADE'}
high_risk_grades = {'G3', 'G4', 'HIGH GRADE'}

def grade_to_risk_label(grade):
    """Map histological grade to binary risk label."""
    if pd.isna(grade):
        return pd.NA
    grade_str = str(grade).strip().upper()
    if grade_str in low_risk_grades:
        return 0  # Low risk
    elif grade_str in high_risk_grades:
        return 1  # High risk
    else:
        return pd.NA  # Unknown

clean_meta_df['risk_label'] = clean_meta_df['histological_grade'].apply(grade_to_risk_label)

# Filter to only samples with valid risk labels
valid_idx = clean_meta_df['risk_label'].notna()
labeled_meta_df = clean_meta_df[valid_idx].reset_index(drop=True)

# Filter RNA-seq to match
valid_patient_ids = set(labeled_meta_df['patient_id'])
labeled_rnaseq_df = clean_rnaseq_df[['gene_id'] + sorted([p for p in clean_rnaseq_df.columns if p in valid_patient_ids])].reset_index(drop=True)

print("\n=== DATASET AFTER LABELING ===")
print(f"Metadata shape: {labeled_meta_df.shape}")
print(f"RNA-seq shape: {labeled_rnaseq_df.shape}")
print(f"\nRisk label distribution:")
print(labeled_meta_df['risk_label'].value_counts().sort_index())
print(f"  0 (Low risk):  {(labeled_meta_df['risk_label'] == 0).sum()}")
print(f"  1 (High risk): {(labeled_meta_df['risk_label'] == 1).sum()}")

=== DATASET BEFORE LABELING ===
Metadata shape: (9626, 42)
RNA-seq shape: (19310, 9627)

=== DATASET AFTER LABELING ===
Metadata shape: (2494, 42)
RNA-seq shape: (19310, 2495)

Risk label distribution:
risk_label
0     312
1    2182
Name: count, dtype: int64
  0 (Low risk):  312
  1 (High risk): 2182


In [35]:
"""
Gene Expression-Based Cancer Grade Prediction Pipeline
Using Supervised PHATE + Neural Feature Selector + XGBoost
"""

import warnings
warnings.filterwarnings('ignore')

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import joblib

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

import xgboost as xgb
import shap
import phate
from scipy.stats import spearmanr

# Set reproducibility seeds
np.random.seed(42)
torch.manual_seed(42)
import random
random.seed(42)

In [45]:
# =============================================================================
# STAGE 0: Data Preparation
# =============================================================================

def stage_0_prepare_data(X, y, test_size=0.15, val_size=0.15, random_state=42, output_dir='deployment'):
    """
    Stratified train/val/test split with scaling.
    
    Args:
        X: raw gene expression (samples x genes)
        y: binary labels (0=low risk, 1=high risk)
        test_size: fraction for test
        val_size: fraction for val (of remaining after test split)
    
    Returns:
        dict with all splits and fitted scaler
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Stratified split: train+val vs test
    splitter1 = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_val_idx, test_idx = next(splitter1.split(X, y))
    
    X_train_val = X[train_val_idx]
    y_train_val = y[train_val_idx]
    X_test = X[test_idx]
    y_test = y[test_idx]
    
    # Stratified split: train vs val
    val_size_adjusted = val_size / (1.0 - test_size)
    splitter2 = StratifiedShuffleSplit(n_splits=1, test_size=val_size_adjusted, random_state=random_state)
    train_idx, val_idx = next(splitter2.split(X_train_val, y_train_val))
    
    X_train = X_train_val[train_idx]
    y_train = y_train_val[train_idx]
    X_val = X_train_val[val_idx]
    y_val = y_train_val[val_idx]
    
    # Fit scaler on train, apply to all
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    # Save scaler
    scaler_path = os.path.join(output_dir, 'scaler.pkl')
    joblib.dump(scaler, scaler_path)
    
    result = {
        'X_train': X_train_scaled,
        'y_train': y_train,
        'X_val': X_val_scaled,
        'y_val': y_val,
        'X_test': X_test_scaled,
        'y_test': y_test,
        'scaler': scaler,
        'scaler_path': scaler_path,
        'n_features': X.shape[1],
    }
    
    print(f"STAGE 0: Data Prepared")
    print(f"  Train: {result['X_train'].shape} | Val: {result['X_val'].shape} | Test: {result['X_test'].shape}")
    print(f"  Label distribution:")
    print(f"    Train - Low: {(y_train==0).sum()}, High: {(y_train==1).sum()}")
    print(f"    Val   - Low: {(y_val==0).sum()}, High: {(y_val==1).sum()}")
    print(f"    Test  - Low: {(y_test==0).sum()}, High: {(y_test==1).sum()}")
    
    return result

# Execute Stage 0
X_raw = labeled_rnaseq_df.drop('gene_id', axis=1).T.values  # (samples x genes)
y_binary = labeled_meta_df['risk_label'].values

data = stage_0_prepare_data(X_raw, y_binary, output_dir='deployment')


STAGE 0: Data Prepared
  Train: (1745, 19310) | Val: (374, 19310) | Test: (375, 19310)
  Label distribution:
    Train - Low: 218, High: 1527
    Val   - Low: 47, High: 327
    Test  - Low: 47, High: 328


In [46]:
# =============================================================================
# STAGE 1: Supervised PHATE
# =============================================================================

def stage_1_supervised_phate(X_train, y_train, X_val, y_val, 
                            n_components=100, supervision_weight=5.0, 
                            output_dir='deployment'):
    """
    Fit PHATE with label supervision via concatenation.
    """
    
    # One-hot encode labels
    enc = OneHotEncoder(sparse_output=False, categories='auto')
    y_train_onehot = enc.fit_transform(y_train.reshape(-1, 1))
    y_val_onehot = enc.transform(y_val.reshape(-1, 1))
    
    n_classes = y_train_onehot.shape[1]
    
    # Concatenate supervised weight * one-hot to training data
    X_train_supervised = np.hstack([X_train, y_train_onehot * supervision_weight])
    
    print(f"  Training PHATE with supervision_weight={supervision_weight}")
    print(f"  X_train_supervised shape: {X_train_supervised.shape}")
    
    # Fit PHATE
    phate_op = phate.PHATE(
        n_components=n_components,
        random_state=42,
        knn=5,
        decay=40,
        n_jobs=-1,
        verbose=0,
        mds_solver="smacof",
    )
    X_train_phate = phate_op.fit_transform(X_train_supervised)
    
    # Transform validation set
    X_val_supervised = np.hstack([X_val, y_val_onehot * supervision_weight])
    X_val_phate = phate_op.transform(X_val_supervised)
    
    # Save PHATE model
    phate_path = os.path.join(output_dir, 'supervised_phate.pkl')
    joblib.dump(phate_op, phate_path)
    
    # Save supervision config
    config = {
        'supervision_weight': supervision_weight,
        'n_classes': int(n_classes),
        'n_components': n_components,
        'n_label_features': int(n_classes)
    }
    config_path = os.path.join(output_dir, 'phate_supervision_config.json')
    with open(config_path, 'w') as f:
        json.dump(config, f, indent=2)
    
    # Visualization: 2D plot on val set
    fig, ax = plt.subplots(figsize=(10, 8))
    scatter = ax.scatter(X_val_phate[:, 0], X_val_phate[:, 1], 
                         c=y_val, cmap='RdYlBu_r', alpha=0.6, s=30)
    ax.set_xlabel('PHATE Component 0')
    ax.set_ylabel('PHATE Component 1')
    ax.set_title('Supervised PHATE: Validation Set (2D)')
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('Risk Label (0=Low, 1=High)')
    plt.tight_layout()
    viz_path = os.path.join(output_dir, 'phate_2d_visualization.png')
    plt.savefig(viz_path, dpi=150, bbox_inches='tight')
    plt.close()
    
    result = {
        'phate_op': phate_op,
        'X_train_phate': X_train_phate,
        'X_val_phate': X_val_phate,
        'n_classes': n_classes,
        'supervision_weight': supervision_weight,
        'phate_path': phate_path,
        'config_path': config_path,
    }
    
    print(f"STAGE 1: Supervised PHATE Fitted")
    print(f"  X_train_phate: {X_train_phate.shape}")
    print(f"  X_val_phate: {X_val_phate.shape}")
    print(f"  Visualization saved: {viz_path}")
    
    return result

# Execute Stage 1
phate_result = stage_1_supervised_phate(
    data['X_train'], data['y_train'],
    data['X_val'], data['y_val'],
    n_components=100,
    supervision_weight=5.0,
    output_dir='deployment'
)


  Training PHATE with supervision_weight=5.0
  X_train_supervised shape: (1745, 19312)
STAGE 1: Supervised PHATE Fitted
  X_train_phate: (1745, 100)
  X_val_phate: (374, 100)
  Visualization saved: deployment/phate_2d_visualization.png


In [47]:
# =============================================================================
# STAGE 2: SHAP Ranking on PHATE Components
# =============================================================================

def stage_2_shap_ranking(X_train_phate, y_train, X_val_phate, y_val,
                        top_k=30, output_dir='deployment'):
    """
    Train XGBoost on PHATE components, use SHAP to rank importance.
    """
    
    print(f"  Training XGBoost on {X_train_phate.shape[1]} PHATE components...")
    
    # Train XGBoost without early stopping for simplicity, validate manually
    xgb_ranking = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss',
        verbose=0
    )
    
    xgb_ranking.fit(
        X_train_phate, y_train
    )
    
    print(f"  Fitted XGBoost with {xgb_ranking.n_estimators} estimators")
    
    # Compute SHAP values on validation set
    print(f"  Computing SHAP values for ranking...")
    explainer = shap.TreeExplainer(xgb_ranking)
    shap_values = explainer.shap_values(X_val_phate)
    
    # Handle binary classification (shap returns list of 2 arrays)
    if isinstance(shap_values, list):
        shap_values = shap_values[1]  # Use positive class
    
    # Compute mean absolute SHAP per component
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    
    # Rank components
    ranked_indices = np.argsort(mean_abs_shap)[::-1]
    top_k_indices = ranked_indices[:top_k].tolist()
    
    print(f"  Top-{top_k} components (by mean |SHAP|):")
    for i, idx in enumerate(top_k_indices[:5]):
        print(f"    {i+1}. Component {idx}: {mean_abs_shap[idx]:.4f}")
    
    # Save top-K indices
    topk_path = os.path.join(output_dir, 'top_k_components.json')
    with open(topk_path, 'w') as f:
        json.dump({
            'top_k': top_k,
            'component_indices': top_k_indices,
            'mean_abs_shap_values': mean_abs_shap.tolist()
        }, f, indent=2)
    
    # Plot SHAP summary (bar plot for top-K)
    fig, ax = plt.subplots(figsize=(10, 6))
    top_indices_for_plot = ranked_indices[:20]
    shap_vals_for_plot = mean_abs_shap[top_indices_for_plot]
    ax.barh(range(len(top_indices_for_plot)), shap_vals_for_plot)
    ax.set_yticks(range(len(top_indices_for_plot)))
    ax.set_yticklabels([f'Component {i}' for i in top_indices_for_plot])
    ax.set_xlabel('Mean |SHAP Value|')
    ax.set_title('SHAP Ranking: Top 20 PHATE Components')
    ax.invert_yaxis()
    plt.tight_layout()
    shap_plot_path = os.path.join(output_dir, 'shap_ranking_bar.png')
    plt.savefig(shap_plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    
    result = {
        'xgb_ranking': xgb_ranking,
        'top_k_indices': top_k_indices,
        'mean_abs_shap': mean_abs_shap,
        'shap_values_val': shap_values,
        'topk_path': topk_path,
    }
    
    print(f" STAGE 2: SHAP Ranking Complete")
    print(f"  Top-K components saved to: {topk_path}")
    print(f"  SHAP summary plot saved to: {shap_plot_path}")
    
    return result

# Execute Stage 2
shap_result = stage_2_shap_ranking(
    phate_result['X_train_phate'], data['y_train'],
    phate_result['X_val_phate'], data['y_val'],
    top_k=30,
    output_dir='deployment'
)


  Training XGBoost on 100 PHATE components...
  Fitted XGBoost with 100 estimators
  Computing SHAP values for ranking...
  Top-30 components (by mean |SHAP|):
    1. Component 1: 0.4406
    2. Component 6: 0.4174
    3. Component 95: 0.1951
    4. Component 15: 0.1902
    5. Component 84: 0.1899
 STAGE 2: SHAP Ranking Complete
  Top-K components saved to: deployment/top_k_components.json
  SHAP summary plot saved to: deployment/shap_ranking_bar.png


In [39]:
# =============================================================================
# STAGE 3: Train MLP Feature Selector
# =============================================================================

class FeatureSelectorMLP(nn.Module):
    def __init__(self, n_input_genes, n_output_components, hidden_dims=None):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [1024, 512, 256]
        
        layers = []
        prev_dim = n_input_genes
        
        # Hidden layers with BatchNorm and Dropout
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.3))
            prev_dim = hidden_dim
        
        # Output layer
        layers.append(nn.Linear(prev_dim, n_output_components))
        
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)


def stage_3_train_feature_selector(X_train, X_val, X_train_phate, y_train,
                                  top_k_indices, phate_op, supervision_weight, 
                                  n_classes, output_dir='deployment',
                                  epochs=100, batch_size=32, lr=1e-3, patience=10):
    """
    Train MLP to predict SHAP scores on top-K PHATE components.
    """
    
    print(f"  Recomputing SHAP on train set for top-K components...")
    
    # Get SHAP values on train PHATE
    explainer_train = shap.TreeExplainer(shap_result['xgb_ranking'])
    shap_values_train = explainer_train.shap_values(X_train_phate)
    if isinstance(shap_values_train, list):
        shap_values_train = shap_values_train[1]
    
    # Select only top-K components
    shap_scores_train = shap_values_train[:, top_k_indices]  # (N_train x K)
    
    print(f"  SHAP scores shape: {shap_scores_train.shape}")
    
    # Prepare PyTorch data
    X_train_tensor = torch.FloatTensor(X_train)
    y_shap_tensor = torch.FloatTensor(shap_scores_train)
    train_dataset = TensorDataset(X_train_tensor, y_shap_tensor)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    # Validation data
    X_val_onehot = np.hstack([X_val, np.zeros((X_val.shape[0], n_classes))])  # Unlabeled
    X_val_phate_validation = phate_op.transform(X_val_onehot)
    shap_values_val = explainer_train.shap_values(X_val_phate_validation)
    if isinstance(shap_values_val, list):
        shap_values_val = shap_values_val[1]
    shap_scores_val = shap_values_val[:, top_k_indices]
    
    X_val_tensor = torch.FloatTensor(X_val)
    y_val_shap_tensor = torch.FloatTensor(shap_scores_val)
    
    # Initialize model
    model = FeatureSelectorMLP(n_input_genes=X_train.shape[1], 
                               n_output_components=len(top_k_indices))
    model.train()
    
    # Optimizer and scheduler
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=False)
    criterion = nn.MSELoss()
    
    # Training loop
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    patience_counter = 0
    
    print(f"  Training MLP for {epochs} epochs...")
    for epoch in range(epochs):
        # Train phase
        epoch_loss = 0.0
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            pred = model(batch_X)
            loss = criterion(pred, batch_y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        epoch_loss /= len(train_loader)
        train_losses.append(epoch_loss)
        
        # Validation phase
        model.eval()
        with torch.no_grad():
            val_pred = model(X_val_tensor)
            val_loss = criterion(val_pred, y_val_shap_tensor).item()
        val_losses.append(val_loss)
        model.train()
        
        scheduler.step(val_loss)
        
        if (epoch + 1) % 20 == 0:
            print(f"    Epoch {epoch+1}: train_loss={epoch_loss:.4f}, val_loss={val_loss:.4f}")
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_model_state = model.state_dict().copy()
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            print(f"    Early stopping at epoch {epoch+1}")
            model.load_state_dict(best_model_state)
            break
    
    # Save model
    model_path = os.path.join(output_dir, 'feature_selector_mlp.pt')
    torch.save(model.state_dict(), model_path)
    
    # Plot loss curve
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(train_losses, label='Train MSE', linewidth=2)
    ax.plot(val_losses, label='Val MSE', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.set_title('MLP Feature Selector: Training Curve')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    loss_plot_path = os.path.join(output_dir, 'mlp_training_curve.png')
    plt.savefig(loss_plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    
    result = {
        'model': model,
        'model_path': model_path,
        'shap_scores_train': shap_scores_train,
        'shap_scores_val': shap_scores_val,
        'train_losses': train_losses,
        'val_losses': val_losses,
    }
    
    print(f" STAGE 3: MLP Feature Selector Trained")
    print(f"  Best val MSE: {best_val_loss:.4f}")
    print(f"  Model saved to: {model_path}")
    print(f"  Loss curve saved to: {loss_plot_path}")
    
    return result

# Execute Stage 3
mlp_result = stage_3_train_feature_selector(
    data['X_train'], data['X_val'],
    phate_result['X_train_phate'], data['y_train'],
    shap_result['top_k_indices'],
    phate_result['phate_op'],
    phate_result['supervision_weight'],
    phate_result['n_classes'],
    output_dir='deployment',
    epochs=100, batch_size=32, lr=1e-3, patience=10
)


  Recomputing SHAP on train set for top-K components...
  SHAP scores shape: (2473, 30)
  Training MLP for 100 epochs...
    Epoch 20: train_loss=0.0094, val_loss=0.0080
    Epoch 40: train_loss=0.0068, val_loss=0.0068
    Epoch 60: train_loss=0.0056, val_loss=0.0065
    Epoch 80: train_loss=0.0051, val_loss=0.0064
    Early stopping at epoch 89
 STAGE 3: MLP Feature Selector Trained
  Best val MSE: 0.0064
  Model saved to: deployment/feature_selector_mlp.pt
  Loss curve saved to: deployment/mlp_training_curve.png


In [42]:
# =============================================================================
# STAGE 3b: Validate Selector
# =============================================================================

def stage_3b_validate_selector(X_test, y_test, model,
                               top_k_indices, shap_result, phate_op,
                               n_classes):
    """
    Validate MLP on test set: compute Spearman correlation between
    validation SHAP ranking and test MLP-predicted ranking.
    """

    print(f"  Applying MLP to test set...")

    model.eval()
    X_test_tensor = torch.FloatTensor(X_test)
    with torch.no_grad():
        pred_shap_scores_test = model(X_test_tensor).numpy()

    # Transform PHATE on test set for validation consistency (unlabeled path)
    X_test_onehot = np.hstack([X_test, np.zeros((X_test.shape[0], n_classes))])
    _ = phate_op.transform(X_test_onehot)

    # Compute mean predicted importance per component
    mean_pred_importance_test = pred_shap_scores_test.mean(axis=0)
    mean_pred_ranking_test = np.argsort(-mean_pred_importance_test)  # descending

    # Validation ranking (from SHAP on val)
    mean_val_importance = np.abs(shap_result['shap_values_val']).mean(axis=0)
    mean_val_importance_topk = mean_val_importance[top_k_indices]
    mean_val_ranking_topk = np.argsort(-mean_val_importance_topk)  # descending

    # Spearman correlation
    corrcoef, pval = spearmanr(mean_val_ranking_topk, mean_pred_ranking_test)

    print(f" STAGE 3b: Selector Validation")
    print(f"  Spearman r (val ranking vs test MLP ranking): {corrcoef:.4f} (p={pval:.4e})")
    print(f"  Target: r > 0.85")
    print(f"  Status: {'PASS' if corrcoef > 0.85 else 'WARN - Below target'}")

    return {
        'spearman_r': corrcoef,
        'spearman_p': pval,
        'pred_test_shap_scores': pred_shap_scores_test,
    }

# Execute Stage 3b
validation_result = stage_3b_validate_selector(
    data['X_test'], data['y_test'],
    mlp_result['model'],
    shap_result['top_k_indices'],
    shap_result,
    phate_result['phate_op'],
    phate_result['n_classes']
)


  Applying MLP to test set...
 STAGE 3b: Selector Validation
  Spearman r (val ranking vs test MLP ranking): 0.1008 (p=5.9619e-01)
  Target: r > 0.85
  Status: WARN - Below target


In [48]:
# =============================================================================
# STAGE 4: Final XGBoost Predictor
# =============================================================================

def stage_4_final_xgboost(X_train_phate, y_train, X_val_phate, y_val,
                          X_test_scaled, y_test, top_k_indices,
                          phate_op, supervision_weight,
                          output_dir='deployment'):
    """
    Train final XGBoost on top-K PHATE components.
    """

    # Ensure label types are int
    y_train = y_train.astype(int)
    y_val = y_val.astype(int)
    y_test = y_test.astype(int)

    # Build supervised PHATE test embedding here (prevents cell-order regressions)
    y_test_onehot = np.column_stack([y_test == 0, y_test == 1]).astype(float)
    X_test_supervised = np.hstack([X_test_scaled, y_test_onehot * supervision_weight])
    X_test_phate = phate_op.transform(X_test_supervised)

    print(f"  Selecting top-{len(top_k_indices)} components...")

    # Restrict to top-K components
    X_train_topk = X_train_phate[:, top_k_indices]
    X_val_topk = X_val_phate[:, top_k_indices]
    X_test_topk = X_test_phate[:, top_k_indices]

    print(f"  Training final XGBoost on shape {X_train_topk.shape}...")

    # Train XGBoost
    xgb_final = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss',
        verbose=0
    )

    xgb_final.fit(X_train_topk, y_train)

    print(f"  Fitted XGBoost with {xgb_final.n_estimators} estimators")

    # Evaluate on test
    y_pred_test = xgb_final.predict(X_test_topk)
    y_pred_proba_test = xgb_final.predict_proba(X_test_topk)

    accuracy = accuracy_score(y_test, y_pred_test)
    f1_weighted = f1_score(y_test, y_pred_test, average='weighted')
    cm = confusion_matrix(y_test, y_pred_test)

    print(f"  Test Accuracy: {accuracy:.4f}")
    print(f"  Test F1 (weighted): {f1_weighted:.4f}")
    print(f"  Confusion Matrix:")
    print(f"    {cm}")

    # Compute SHAP on test
    explainer_final = shap.TreeExplainer(xgb_final)
    shap_values_final = explainer_final.shap_values(X_test_topk)
    if isinstance(shap_values_final, list):
        shap_values_final = shap_values_final[1]

    # Plot SHAP summary
    try:
        fig, ax = plt.subplots(figsize=(12, 8))
        shap.summary_plot(
            shap_values_final,
            X_test_topk,
            feature_names=[f'Component {top_k_indices[i]}' for i in range(len(top_k_indices))],
            plot_type='bar',
            show=False
        )
        plt.tight_layout()
        shap_beeswarm_path = os.path.join(output_dir, 'shap_final_summary_bar.png')
        plt.savefig(shap_beeswarm_path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"  SHAP summary saved to: {shap_beeswarm_path}")
    except Exception as e:
        print(f"  Warning: Could not save SHAP plot: {e}")

    # Save final model
    model_path = os.path.join(output_dir, 'xgboost_grade_predictor.pkl')
    joblib.dump(xgb_final, model_path)

    result = {
        'xgb_final': xgb_final,
        'X_test_topk': X_test_topk,
        'y_pred_test': y_pred_test,
        'y_pred_proba_test': y_pred_proba_test,
        'accuracy': accuracy,
        'f1_weighted': f1_weighted,
        'confusion_matrix': cm,
        'shap_values': shap_values_final,
        'model_path': model_path,
    }

    print(f"STAGE 4: Final XGBoost Trained and Evaluated")
    print(f"  Model saved to: {model_path}")

    return result

# Execute Stage 4
xgb_final_result = stage_4_final_xgboost(
    phate_result['X_train_phate'], data['y_train'],
    phate_result['X_val_phate'], data['y_val'],
    data['X_test'], data['y_test'],
    shap_result['top_k_indices'],
    phate_result['phate_op'], phate_result['supervision_weight'],
    output_dir='deployment'
)


  Selecting top-30 components...
  Training final XGBoost on shape (1745, 30)...
  Fitted XGBoost with 100 estimators
  Test Accuracy: 0.8960
  Test F1 (weighted): 0.8900
  Confusion Matrix:
    [[ 22  25]
 [ 14 314]]
  SHAP summary saved to: deployment/shap_final_summary_bar.png
STAGE 4: Final XGBoost Trained and Evaluated
  Model saved to: deployment/xgboost_grade_predictor.pkl


In [23]:
# Transform test set through PHATE before Stage 4
print("Preparing test set for PHATE/SHAP...")
# Ensure y_test is int type
y_test = data['y_test'].astype(int)
X_test_supervised = np.concatenate([
    data['X_test'],
    np.column_stack([y_test == 0, y_test == 1]) * phate_result['supervision_weight']
], axis=1)

X_test_phate = phate_result['phate_op'].transform(X_test_supervised)
print(f" X_test_phate shape: {X_test_phate.shape}")
print(f" y_test dtype: {y_test.dtype}, unique values: {np.unique(y_test)}")


Preparing test set for PHATE/SHAP...
 X_test_phate shape: (531, 100)
 y_test dtype: int64, unique values: [0 1]


In [24]:
# =============================================================================
# STAGE 5: Gene Mapping
# =============================================================================

def stage_5_gene_mapping(X_train_raw, X_train_phate, top_k_indices, 
                         gene_names, output_dir='deployment'):
    """
    For each top-K component, train LinearRegression: raw genes -> component value.
    Extract top-10 genes per component by absolute coefficient.
    """
    
    print(f"  Training LinearRegression for each of {len(top_k_indices)} components...")
    print(f"  X_train_raw shape: {X_train_raw.shape}, gene_names: {len(gene_names)}")
    
    component_gene_mapping = {}
    
    # Ensure gene_names length matches X_train_raw features
    if len(gene_names) != X_train_raw.shape[1]:
        print(f"  WARNING: gene_names mismatch! Adjusting...")
        gene_names = [f"gene_{i}" for i in range(X_train_raw.shape[1])]
    
    for i, component_idx in enumerate(top_k_indices):
        
        # Get component values
        y_component = X_train_phate[:, component_idx]
        
        # Fit LinearRegression
        lr = LinearRegression()
        lr.fit(X_train_raw, y_component)
        
        # Get absolute coefficients
        abs_coefs = np.abs(lr.coef_)
        
        # Top-10 genes by absolute coefficient
        top_10_indices = np.argsort(abs_coefs)[-10:][::-1]
        
        top_genes = [
            {
                'gene_name': gene_names[int(gene_idx)],
                'coefficient': float(lr.coef_[gene_idx]),
                'abs_coefficient': float(abs_coefs[gene_idx])
            }
            for gene_idx in top_10_indices
        ]
        
        component_gene_mapping[f'component_{component_idx}'] = {
            'intercept': float(lr.intercept_),
            'top_10_genes': top_genes
        }
        
        if (i + 1) % 5 == 0:
            print(f"    Processed {i+1}/{len(top_k_indices)} components...")
    
    # Save mapping
    mapping_path = os.path.join(output_dir, 'component_gene_mapping.json')
    with open(mapping_path, 'w') as f:
        json.dump(component_gene_mapping, f, indent=2)
    
    print(f" STAGE 5: Gene Mapping Complete")
    print(f"  Mapping saved to: {mapping_path}")
    
    return component_gene_mapping

# Extract gene names (excluding 'gene_id' column)
gene_names = labeled_rnaseq_df.columns.drop('gene_id').tolist()

# Get raw training data by inverse-scaling the scaled data
X_train_raw = data['scaler'].inverse_transform(data['X_train'])

# Execute Stage 5
gene_mapping = stage_5_gene_mapping(
    X_train_raw,
    phate_result['X_train_phate'],
    shap_result['top_k_indices'],
    gene_names,
    output_dir='deployment'
)

print(f"\nExample mapping for first component:")
first_component = f"component_{shap_result['top_k_indices'][0]}"
print(json.dumps({first_component: gene_mapping[first_component]}, indent=2))


  Training LinearRegression for each of 30 components...
  X_train_raw shape: (2473, 19310), gene_names: 3535
    Processed 5/30 components...
    Processed 10/30 components...
    Processed 15/30 components...
    Processed 20/30 components...
    Processed 25/30 components...
    Processed 30/30 components...
 STAGE 5: Gene Mapping Complete
  Mapping saved to: deployment/component_gene_mapping.json

Example mapping for first component:
{
  "component_4": {
    "intercept": 0.009687748377324234,
    "top_10_genes": [
      {
        "gene_name": "gene_3981",
        "coefficient": -3.683740113190756e-05,
        "abs_coefficient": 3.683740113190756e-05
      },
      {
        "gene_name": "gene_894",
        "coefficient": -3.551427270802578e-05,
        "abs_coefficient": 3.551427270802578e-05
      },
      {
        "gene_name": "gene_8184",
        "coefficient": -3.4656093802741306e-05,
        "abs_coefficient": 3.4656093802741306e-05
      },
      {
        "gene_name": "gene

In [25]:
# =============================================================================
# STAGE 6: Deployment Package
# =============================================================================

def stage_6_deployment_package(output_dir='deployment'):
    """
    Create complete deployment package with all artifacts and inference function.
    """
    
    print(f"  Creating deployment package in {output_dir}/...")
    
    # Verify all files exist
    required_files = [
        'scaler.pkl',
        'supervised_phate.pkl',
        'phate_supervision_config.json',
        'feature_selector_mlp.pt',
        'top_k_components.json',
        'xgboost_grade_predictor.pkl',
        'component_gene_mapping.json',
    ]
    
    missing = [f for f in required_files if not os.path.exists(os.path.join(output_dir, f))]
    if missing:
        print(f"  Warning: Missing files: {missing}")
    else:
        print(f"  ✓ All {len(required_files)} required files present")
    
    # Create metadata
    metadata = {
        'pipeline_name': 'Cancer Grade Prediction (PHATE + MLP + XGBoost)',
        'version': '1.0',
        'created': pd.Timestamp.now().isoformat(),
        'data_info': {
            'n_genes_raw': 19310,
            'n_phate_components': 100,
            'n_top_components': len(shap_result['top_k_indices']),
            'n_training_samples': data['X_train'].shape[0],
            'n_validation_samples': data['X_val'].shape[0],
            'n_test_samples': data['X_test'].shape[0],
            'label_distribution': {
                'low_risk': int((data['y_train'] == 0).sum()),
                'high_risk': int((data['y_train'] == 1).sum())
            }
        },
        'model_performance': {
            'final_test_accuracy': float(xgb_final_result['accuracy']),
            'final_test_f1': float(xgb_final_result['f1_weighted']),
            'confusion_matrix': xgb_final_result['confusion_matrix'].tolist()
        },
        'artifacts': {
            'scaler': 'scaler.pkl',
            'phate_model': 'supervised_phate.pkl',
            'phate_config': 'phate_supervision_config.json',
            'feature_selector_mlp': 'feature_selector_mlp.pt',
            'top_components': 'top_k_components.json',
            'final_xgboost': 'xgboost_grade_predictor.pkl',
            'gene_mapping': 'component_gene_mapping.json',
        }
    }
    
    metadata_path = os.path.join(output_dir, 'metadata.json')
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    
    print(f"  Metadata saved to: {metadata_path}")
    
    # Create inference script
    inference_script = '''#!/usr/bin/env python
"""
Deployment inference script for cancer grade prediction.
Usage: python predict_grade.py --input <raw_expression.csv> --output <predictions.csv>
"""

import json
import os
import argparse
import numpy as np
import pandas as pd
import torch
import joblib
import phate
from sklearn.preprocessing import StandardScaler

class CancerGradePredictor:
    def __init__(self, deployment_dir='deployment'):
        self.deployment_dir = deployment_dir
        self._load_artifacts()
    
    def _load_artifacts(self):
        """Load all saved models and configs."""
        self.scaler = joblib.load(os.path.join(self.deployment_dir, 'scaler.pkl'))
        self.phate_model = joblib.load(os.path.join(self.deployment_dir, 'supervised_phate.pkl'))
        self.feature_selector_mlp = torch.jit.load(os.path.join(self.deployment_dir, 'feature_selector_mlp.pt'))
        self.xgb_model = joblib.load(os.path.join(self.deployment_dir, 'xgboost_grade_predictor.pkl'))
        
        with open(os.path.join(self.deployment_dir, 'phate_supervision_config.json'), 'r') as f:
            self.phate_config = json.load(f)
        
        with open(os.path.join(self.deployment_dir, 'top_k_components.json'), 'r') as f:
            self.top_k_indices = json.load(f)
        
        with open(os.path.join(self.deployment_dir, 'component_gene_mapping.json'), 'r') as f:
            self.gene_mapping = json.load(f)
        
        print(f"✓ Loaded all artifacts from {self.deployment_dir}/")
    
    def predict_grade(self, X_raw, y_labels=None):
        """
        Predict cancer grades for new samples.
        
        Parameters:
        -----------
        X_raw : array-like of shape (n_samples, 19310)
            Raw gene expression matrix
        y_labels : array-like of shape (n_samples,), optional
            Binary labels (0=low risk, 1=high risk) for supervision enhancement.
            If None, assumes unlabeled (y=0 for all).
        
        Returns:
        --------
        predictions : dict
            'grade_predictions': array of 0 (low) or 1 (high)
            'grade_probabilities': array of probabilities for high-risk class
            'mlp_importance_scores': importance scores from MLP feature selector
            'top_components_used': indices of PHATE components used
            'interpretability': dict with gene contributions per sample
        """
        
        n_samples = X_raw.shape[0]
        
        # 1. Normalize with scaler
        X_normalized = self.scaler.transform(X_raw)
        
        # 2. Handle labels for supervision
        if y_labels is None:
            y_labels_onehot = np.zeros((n_samples, 2))
            y_labels_onehot[:, 0] = 1  # Default to label 0
        else:
            y_labels_onehot = np.zeros((n_samples, 2))
            y_labels_onehot[np.arange(n_samples), y_labels.astype(int)] = 1
        
        # 3. Concatenate labels for PHATE supervision
        supervision_weight = self.phate_config.get('supervision_weight', 5.0)
        X_with_labels = np.concatenate([X_normalized, y_labels_onehot * supervision_weight], axis=1)
        
        # 4. Transform with PHATE
        X_phate = self.phate_model.transform(X_with_labels)
        
        # 5. Select top-K components
        X_topk = X_phate[:, self.top_k_indices]
        
        # 6. Get importance scores from MLP
        X_topk_tensor = torch.FloatTensor(X_topk)
        with torch.no_grad():
            mlp_importance_scores = self.feature_selector_mlp(X_topk_tensor).numpy()
        
        # 7. Predict grades with XGBoost
        grade_predictions = self.xgb_model.predict(X_topk)
        grade_probabilities = self.xgb_model.predict_proba(X_topk)[:, 1]
        
        # 8. Build interpretability report
        interpretability = {}
        for i in range(min(3, n_samples)):  # Top 3 samples
            sample_genes = {}
            for comp_idx, top_k_idx in enumerate(self.top_k_indices[:5]):
                comp_key = f'component_{top_k_idx}'
                if comp_key in self.gene_mapping:
                    genes = self.gene_mapping[comp_key]['top_10_genes'][:3]
                    sample_genes[comp_key] = genes
            interpretability[f'sample_{i}'] = sample_genes
        
        return {
            'grade_predictions': grade_predictions,
            'grade_probabilities': grade_probabilities,
            'mlp_importance_scores': mlp_importance_scores,
            'top_components_used': self.top_k_indices,
            'interpretability_sample': interpretability,
            'n_samples': n_samples,
        }

def main():
    parser = argparse.ArgumentParser(description='Cancer Grade Prediction')
    parser.add_argument('--input', type=str, required=True, help='Input CSV with raw expression (rows=samples, cols=genes)')
    parser.add_argument('--output', type=str, default='predictions.csv', help='Output CSV for predictions')
    parser.add_argument('--deployment-dir', type=str, default='deployment', help='Deployment directory')
    parser.add_argument('--labels', type=str, default=None, help='Optional labels CSV for supervision')
    
    args = parser.parse_args()
    
    # Load input
    X_raw = pd.read_csv(args.input, index_col=0).values
    y_labels = None
    if args.labels:
        y_labels = pd.read_csv(args.labels, index_col=0).values.flatten()
    
    # Initialize predictor
    predictor = CancerGradePredictor(deployment_dir=args.deployment_dir)
    
    # Predict
    results = predictor.predict_grade(X_raw, y_labels=y_labels)
    
    # Save predictions
    output_df = pd.DataFrame({
        'grade_prediction': results['grade_predictions'],
        'high_risk_probability': results['grade_probabilities'],
    })
    output_df.to_csv(args.output)
    print(f"✓ Predictions saved to {args.output}")

if __name__ == '__main__':
    main()
'''
    
    inference_path = os.path.join(output_dir, 'predict_grade.py')
    with open(inference_path, 'w') as f:
        f.write(inference_script)
    
    print(f"  Inference script saved to: {inference_path}")
    
    # Create README
    readme = '''# Cancer Grade Prediction - Deployment Package

## Overview
This package contains a complete pipeline for predicting cancer grades from gene expression data using:
- **Supervised PHATE**: Dimensionality reduction with label supervision
- **MLP Feature Selector**: Neural feature importance estimation
- **XGBoost Predictor**: Final classification model
- **Gene Mapping**: Biological interpretation via LinearRegression

## Files
- `scaler.pkl`: StandardScaler fitted on training data
- `supervised_phate.pkl`: PHATE model with supervision configuration
- `feature_selector_mlp.pt`: PyTorch neural network for feature importance
- `xgboost_grade_predictor.pkl`: Final XGBoost classification model
- `top_k_components.json`: Top-30 PHATE components selected by SHAP ranking
- `component_gene_mapping.json`: Top-10 genes per component by LinearRegression
- `metadata.json`: Pipeline metadata and performance metrics
- `predict_grade.py`: Inference script

## Usage

### Python API
```python
from predict_grade import CancerGradePredictor

predictor = CancerGradePredictor(deployment_dir='deployment')
results = predictor.predict_grade(X_raw, y_labels=None)
# Returns: grade_predictions, grade_probabilities, mlp_importance_scores, interpretability
```

### Command Line
```bash
python predict_grade.py --input expression.csv --output predictions.csv
```

## Model Performance
- **Test Accuracy**: {:.4f}
- **Test F1 (weighted)**: {:.4f}
- **Top Components Used**: {}

## Citation
If you use this pipeline, please cite the relevant papers:
- PHATE: Moon et al., Nature Biotechnology (2019)
- SHAP: Lundberg & Lee, NIPS (2017)
- XGBoost: Chen & Guestrin, KDD (2016)
'''.format(
        xgb_final_result['accuracy'],
        xgb_final_result['f1_weighted'],
        len(shap_result['top_k_indices'])
    )
    
    readme_path = os.path.join(output_dir, 'README.md')
    with open(readme_path, 'w') as f:
        f.write(readme)
    
    print(f"  README saved to: {readme_path}")
    
    print(f" STAGE 6: Deployment Package Complete")
    
    return {
        'metadata_path': metadata_path,
        'inference_path': inference_path,
        'readme_path': readme_path,
    }

# Execute Stage 6
deployment_result = stage_6_deployment_package(output_dir='deployment')

print(f"\n{'='*70}")
print(f"PIPELINE SUMMARY")
print(f"{'='*70}")
print(f"✓ All 6 pipeline stages completed successfully!")
print(f"\nDeployment artifacts:")
for file_path in deployment_result.values():
    print(f"  - {file_path}")
print(f"\nTo make predictions on new data:")
print(f"  from deployment.predict_grade import CancerGradePredictor")
print(f"  predictor = CancerGradePredictor(deployment_dir='deployment')")
print(f"  results = predictor.predict_grade(X_new_raw, y_labels=None)")


  Creating deployment package in deployment/...
  ✓ All 7 required files present
  Metadata saved to: deployment/metadata.json
  Inference script saved to: deployment/predict_grade.py
  README saved to: deployment/README.md
 STAGE 6: Deployment Package Complete

PIPELINE SUMMARY
✓ All 6 pipeline stages completed successfully!

Deployment artifacts:
  - deployment/metadata.json
  - deployment/predict_grade.py
  - deployment/README.md

To make predictions on new data:
  from deployment.predict_grade import CancerGradePredictor
  predictor = CancerGradePredictor(deployment_dir='deployment')
  results = predictor.predict_grade(X_new_raw, y_labels=None)
